# Seguimiento SIDCAR Grupo B - solo Oficio

Versión ajustada de la lógica sin romper el flujo que ya veníamos trabajando.

## Ajustes aplicados
- **Solo trae tipo `Oficio`** en ambas bases.
- Mantiene la lógica de:
  - tomar **Radicados** como base inicial
  - intentar cruce con **Preradicados**
  - agregar **preradicados no enlazados** como filas independientes
- deja **todo en una sola hoja**
- incluye:
  - abogado
  - oficio
  - revisión
  - expediente analizado
  - preradicado
  - radicado
  - estados del flujo


In [ ]:

import os
import re
import unicodedata
import numpy as np
import pandas as pd

ELABORADORES = [
    "Ana Maria Guzman Acosta",
    "Diana Carolina Morera Acuña",
    "Andrea Claudia Martin Martinez",
    "Cristian Joaquin Gil Aponte",
    "Laura Stephania Martinez Leandro",
    "Valentina Bernal Triana",
]

OUTPUT_DIR = "output_SIDCAR"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "seguimiento_sidcar,grupo_B.xlsx")
SHEET_NAME = "seguimiento_sidcar_grupo_B"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def norm_text(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = s.upper().strip()
    s = re.sub(r"\s+", " ", s)
    return s

ELABORADORES_N = [norm_text(x) for x in ELABORADORES]

def clean_code(v):
    if pd.isna(v):
        return ""
    s = str(v).strip()
    s = re.sub(r"\.0$", "", s)
    s = re.sub(r"\s+", "", s)
    return s

def extract_expediente(text):
    if pd.isna(text):
        return ""
    txt = str(text)
    patterns = [
        r"(?i)\bexpediente(?!s)\b\s*(?:no\.?|n[°o]?\.?|#|:|-)?\s*([A-Za-z0-9\-/]{3,})",
        r"(?i)\bexp\.?\b\s*(?:no\.?|n[°o]?\.?|#|:|-)?\s*([A-Za-z0-9\-/]{3,})",
        r"(?i)\bexpe\.?\b\s*(?:no\.?|n[°o]?\.?|#|:|-)?\s*([A-Za-z0-9\-/]{3,})",
    ]
    for pat in patterns:
        m = re.search(pat, txt)
        if m:
            return clean_code(m.group(1))
    txt2 = txt.strip()
    if re.fullmatch(r"[A-Za-z0-9\-/]{3,}", txt2):
        return clean_code(txt2)
    return ""

def clean_reviewer_name(s):
    s = "" if pd.isna(s) else str(s).strip()
    if not s or s.lower() == "nan":
        return ""
    s = re.sub(r"\s*/\s*DJA\s*$", "", s, flags=re.I)
    s = re.sub(r"\[[^\]]+\]$", "", s).strip()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def build_headers(df_raw):
    def clean(v):
        if pd.isna(v):
            return ""
        t = str(v).strip()
        t = re.sub(r"\s+", " ", t)
        if t.lower().startswith("unnamed:"):
            return ""
        return t

    headers = []
    for a, b in zip(df_raw.iloc[0], df_raw.iloc[1]):
        a = clean(a)
        b = clean(b)
        if a and b:
            h = a if a.lower() == b.lower() else f"{a} {b}"
        elif a:
            h = a
        elif b:
            h = b
        else:
            h = "columna"
        headers.append(h)

    seen = {}
    unique = []
    for h in headers:
        if h not in seen:
            seen[h] = 0
            unique.append(h)
        else:
            seen[h] += 1
            unique.append(f"{h}_{seen[h]}")
    return unique

def read_sidcar_xlsx(path):
    raw = pd.read_excel(path, header=None)
    df = pd.read_excel(path, header=None, skiprows=2)
    df.columns = build_headers(raw)
    df = df.dropna(how="all").reset_index(drop=True)
    return df

def find_input_file(filename):
    posibles = [
        filename,
        os.path.join("input_SIDCAR", filename),
        os.path.join(os.getcwd(), filename),
        os.path.join(os.getcwd(), "input_SIDCAR", filename),
        os.path.join("/mnt/data", filename),
    ]
    for p in posibles:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"No encontré el archivo {filename}. Revisé: {posibles}")

def is_oficio(series):
    s = series.astype(str).fillna("").map(norm_text)
    return s.str.contains("OFICIO", na=False)


In [ ]:

# Cargar entradas
path_preradicados = find_input_file("preradicados.xlsx")
path_radicados = find_input_file("Radicados.xlsx")

df_preradicados_raw = read_sidcar_xlsx(path_preradicados)
df_radicados_raw = read_sidcar_xlsx(path_radicados)

print("Ruta preradicados:", path_preradicados)
print("Ruta radicados   :", path_radicados)
print("Preradicados shape:", df_preradicados_raw.shape)
print("Radicados shape   :", df_radicados_raw.shape)


In [ ]:

def build_radicados_detail(df):
    df = df.copy()
    df["nombre_persona"] = df["Elaboró"].astype(str).map(norm_text)
    df = df[df["nombre_persona"].isin(ELABORADORES_N)].copy()
    df = df[is_oficio(df["Tipo"])].copy()

    df["expediente"] = ""
    for col in ["Info SAE Expediente", "Asunto", "Título"]:
        if col in df.columns:
            mask = df["expediente"].eq("")
            df.loc[mask, "expediente"] = df.loc[mask, col].apply(extract_expediente)

    df["expediente_analizado"] = df["expediente"].replace("", "SIN EXPEDIENTE DETECTADO")
    df["numero_radicado"] = df["# Radicado"].astype(str).replace("nan", "").str.strip()
    df["numero_preradicado_ref"] = df["Pre-Radicado"].astype(str).replace("nan", "").str.strip()
    df["fecha_radicado"] = pd.to_datetime(df["F Radicado"], errors="coerce")
    df["estado_radicado"] = df["Estado"].astype(str).replace("nan", "").str.strip()
    df["tipo_documento_radicado"] = df["Tipo"].astype(str).replace("nan", "").str.strip()
    df["asunto_radicado"] = df["Asunto"].astype(str).replace("nan", "").str.strip()
    df["revisor"] = df["Revisó"].apply(clean_reviewer_name)
    df["revision_abogado"] = np.where(
        df["revisor"].str.strip() != "",
        "CON REVISION",
        "SIN DATO DE REVISION"
    )

    keep = [
        "nombre_persona", "expediente", "expediente_analizado",
        "numero_radicado", "numero_preradicado_ref", "fecha_radicado",
        "estado_radicado", "tipo_documento_radicado", "asunto_radicado",
        "revisor", "revision_abogado"
    ]
    return df[keep].copy().reset_index(drop=True)

def build_preradicados_detail(df):
    df = df.copy()
    df["nombre_persona"] = df["Usuario"].astype(str).map(norm_text)
    df = df[df["nombre_persona"].isin(ELABORADORES_N)].copy()
    df = df[is_oficio(df["Tipo Documento"])].copy()

    df["expediente"] = df["Asunto"].apply(extract_expediente)
    df["expediente_analizado"] = df["expediente"].replace("", "SIN EXPEDIENTE DETECTADO")
    df["numero_preradicado"] = df["Número"].astype(str).replace("nan", "").str.strip()
    df["fecha_preradicado"] = pd.to_datetime(df["Información de Creación Fecha"], errors="coerce")
    df["estado_preradicado"] = df["Estado"].astype(str).replace("nan", "").str.strip()
    df["tipo_documento_preradicado"] = df["Tipo Documento"].astype(str).replace("nan", "").str.strip()
    df["asunto_preradicado"] = df["Asunto"].astype(str).replace("nan", "").str.strip()
    df["revision_abogado_pre"] = np.where(
        df["estado_preradicado"].astype(str).str.contains("Revisión", case=False, na=False),
        "EN REVISION",
        "SIN REVISION"
    )

    keep = [
        "nombre_persona", "expediente", "expediente_analizado",
        "numero_preradicado", "fecha_preradicado", "estado_preradicado",
        "tipo_documento_preradicado", "asunto_preradicado", "revision_abogado_pre"
    ]
    return df[keep].copy().reset_index(drop=True)

df_radicados_det = build_radicados_detail(df_radicados_raw)
df_preradicados_det = build_preradicados_detail(df_preradicados_raw)

print("Radicados Oficio:", df_radicados_det.shape)
print("Preradicados Oficio:", df_preradicados_det.shape)
display(df_radicados_det.head(10))
display(df_preradicados_det.head(10))


In [ ]:

def build_seguimiento_oficio(rad_det, pre_det):
    pre = pre_det.copy().reset_index(drop=True)
    pre["pre_row_id"] = range(1, len(pre) + 1)

    rad = rad_det.copy().reset_index(drop=True)
    rad["rad_row_id"] = range(1, len(rad) + 1)

    # 1) Merge exacto por referencia de Pre-Radicado
    rad_con_ref = rad[rad["numero_preradicado_ref"].astype(str).str.strip() != ""].copy()
    rad_sin_ref = rad[rad["numero_preradicado_ref"].astype(str).str.strip() == ""].copy()

    merge_ref = rad_con_ref.merge(
        pre,
        left_on=["nombre_persona", "numero_preradicado_ref"],
        right_on=["nombre_persona", "numero_preradicado"],
        how="left",
        suffixes=("", "_pre")
    )

    # 2) Fallback por nombre + expediente, solo si expediente existe
    rad_sin_ref_con_exp = rad_sin_ref[rad_sin_ref["expediente"].astype(str).str.strip() != ""].copy()
    rad_sin_ref_sin_exp = rad_sin_ref[rad_sin_ref["expediente"].astype(str).str.strip() == ""].copy()
    pre_con_exp = pre[pre["expediente"].astype(str).str.strip() != ""].copy()

    merge_exp = rad_sin_ref_con_exp.merge(
        pre_con_exp,
        on=["nombre_persona", "expediente"],
        how="left",
        suffixes=("", "_pre")
    )

    # 3) Radicados sin expediente: se conservan independientes
    for col in pre.columns:
        if col not in rad_sin_ref_sin_exp.columns:
            rad_sin_ref_sin_exp[col] = np.nan

    merged = pd.concat([merge_ref, merge_exp, rad_sin_ref_sin_exp], ignore_index=True, sort=False)

    # 4) Preradicados no enlazados: se agregan como filas independientes
    matched_pre_ids = set(
        merged.loc[merged["pre_row_id"].notna(), "pre_row_id"].astype(int).tolist()
    )

    pre_unmatched = pre[~pre["pre_row_id"].isin(matched_pre_ids)].copy()

    if not pre_unmatched.empty:
        for col in rad.columns:
            if col not in pre_unmatched.columns:
                pre_unmatched[col] = np.nan
        merged = pd.concat([merged, pre_unmatched], ignore_index=True, sort=False)

    # 5) Limpieza de texto
    text_cols = [
        "expediente", "expediente_analizado",
        "numero_radicado", "numero_preradicado_ref", "estado_radicado",
        "tipo_documento_radicado", "asunto_radicado", "revisor", "revision_abogado",
        "numero_preradicado", "estado_preradicado", "tipo_documento_preradicado",
        "asunto_preradicado", "revision_abogado_pre"
    ]
    for c in text_cols:
        if c in merged.columns:
            merged[c] = merged[c].fillna("").astype(str).replace("nan", "").str.strip()

    has_rad = merged["numero_radicado"].str.strip() != ""
    has_pre = merged["numero_preradicado"].str.strip() != ""

    estado_pre_norm = merged["estado_preradicado"].str.upper().apply(
        lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("ascii")
    )

    merged["estado_de_elaborado"] = np.where(
        has_pre,
        merged["estado_preradicado"],
        "SIN PRERADICADO ASOCIADO"
    )

    merged["estado_en_revision"] = np.where(
        estado_pre_norm.str.contains("ENVIADO PARA REVISION", na=False),
        "EN VOBO DEL REVISOR",
        ""
    )

    merged["firmado_terminado"] = np.where(
        has_rad,
        "FINALIZADO",
        ""
    )

    def fase_actual(row):
        if str(row.get("numero_radicado", "")).strip():
            return "FINALIZADO"

        estado = unicodedata.normalize(
            "NFKD",
            str(row.get("estado_preradicado", "")).upper()
        ).encode("ascii", "ignore").decode("ascii")

        if "ENVIADO PARA REVISION" in estado:
            return "EN VOBO DEL REVISOR"
        if "DEVUELTO" in estado:
            return "DEVUELTO"
        if str(row.get("numero_preradicado", "")).strip():
            return "PRERADICADO"

        return "SIN TRAZA"

    merged["fase_actual"] = merged.apply(fase_actual, axis=1)

    def responsable_actual(row):
        if row["fase_actual"] == "FINALIZADO":
            return "FINALIZADO"
        if row["fase_actual"] == "EN VOBO DEL REVISOR":
            rev = str(row.get("revisor", "")).strip()
            return rev if rev else "REVISOR (SIN DATO EN BASE)"
        return row["nombre_persona"]

    merged["responsable_actual"] = merged.apply(responsable_actual, axis=1)

    merged["fecha_inicio"] = pd.to_datetime(merged.get("fecha_preradicado"), errors="coerce")
    merged["fecha_fin"] = pd.to_datetime(merged.get("fecha_radicado"), errors="coerce")

    merged["dias_gestion"] = (
        merged["fecha_fin"].fillna(pd.Timestamp.today().normalize()) - merged["fecha_inicio"]
    ).dt.days
    merged.loc[merged["fecha_inicio"].isna(), "dias_gestion"] = np.nan

    def estado_vencimiento(row):
        if pd.notna(row["fecha_fin"]):
            return "FINALIZADO"
        if pd.isna(row["fecha_inicio"]):
            return "SIN FECHA INICIO"
        d = row["dias_gestion"]
        if pd.isna(d):
            return "SIN FECHA INICIO"
        if d > 30:
            return "VENCIDO"
        if d >= 25:
            return "POR VENCER"
        return "EN TERMINO"

    merged["estado_vencimiento"] = merged.apply(estado_vencimiento, axis=1)

    merged["fuente_registro"] = np.where(
        has_rad & has_pre,
        "RADICADOS + PRERADICADOS",
        np.where(has_rad, "RADICADOS", np.where(has_pre, "PRERADICADOS", "SIN FUENTE"))
    )

    # Columna unificada para Power BI
    merged["tipo_documento"] = np.where(
        has_rad,
        merged["tipo_documento_radicado"],
        merged["tipo_documento_preradicado"]
    )

    merged["revision_abogado"] = np.where(
        has_rad,
        merged["revision_abogado"],
        merged["revision_abogado_pre"]
    )

    merged["expediente_analizado"] = np.where(
        merged["expediente"].astype(str).str.strip() != "",
        merged["expediente"],
        "SIN EXPEDIENTE DETECTADO"
    )

    mapa_nombres = {norm_text(x): x for x in ELABORADORES}
    merged["nombre_persona"] = merged["nombre_persona"].map(lambda x: mapa_nombres.get(x, x.title()))

    columnas_finales = [
        "nombre_persona",
        "tipo_documento",
        "revision_abogado",
        "expediente",
        "expediente_analizado",
        "numero_preradicado",
        "fecha_preradicado",
        "estado_preradicado",
        "tipo_documento_preradicado",
        "asunto_preradicado",
        "numero_radicado",
        "fecha_radicado",
        "estado_radicado",
        "tipo_documento_radicado",
        "asunto_radicado",
        "revisor",
        "estado_de_elaborado",
        "estado_en_revision",
        "firmado_terminado",
        "fase_actual",
        "responsable_actual",
        "fecha_inicio",
        "fecha_fin",
        "dias_gestion",
        "estado_vencimiento",
        "fuente_registro",
    ]

    for c in columnas_finales:
        if c not in merged.columns:
            merged[c] = ""

    df_final = merged[columnas_finales].copy()

    df_final = df_final.sort_values(
        by=["nombre_persona", "expediente", "numero_preradicado", "numero_radicado"],
        ascending=[True, True, True, True]
    ).reset_index(drop=True)

    return df_final

df_seguimiento = build_seguimiento_oficio(df_radicados_det, df_preradicados_det)

print("Salida final:", df_seguimiento.shape)
display(df_seguimiento.head(30))


In [ ]:

# Resumen rápido
print("Registros por fuente:")
display(df_seguimiento["fuente_registro"].value_counts(dropna=False).rename_axis("fuente_registro").reset_index(name="total"))

print("Registros por fase:")
display(df_seguimiento["fase_actual"].value_counts(dropna=False).rename_axis("fase_actual").reset_index(name="total"))

print("Registros por abogado:")
display(df_seguimiento["nombre_persona"].value_counts(dropna=False).rename_axis("nombre_persona").reset_index(name="total"))


In [ ]:

# Exportar una sola hoja
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    df_seguimiento.to_excel(writer, sheet_name=SHEET_NAME, index=False)

print("Archivo generado correctamente en:")
print(OUTPUT_FILE)
